In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the working file — not the big one
df = pd.read_csv('../data/raw/lending_club_working.csv', low_memory=False)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nFirst 5 rows:")
df.head()

Matplotlib is building the font cache; this may take a moment.


Rows: 938,821
Columns: 151

First 5 rows:


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,130954621,NaN,5000.0,5000.0,5000.0,36 months,20.39,186.82,D,D4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,130964697,NaN,15000.0,15000.0,15000.0,36 months,9.92,483.45,B,B2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,130955326,NaN,11200.0,11200.0,11200.0,60 months,30.79,367.82,G,G1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,130504052,NaN,25000.0,25000.0,25000.0,60 months,21.85,688.35,D,D5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,130956066,NaN,3000.0,3000.0,3000.0,36 months,7.34,93.10,A,A4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# See every column, its data type, and how many non-null values it has
print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Dataset Info:
Shape: (938821, 151)

Data types:
float64    114
str         36
int64        1
Name: count, dtype: int64

Memory usage: 2591.7 MB


In [3]:
# Calculate missing percentage for every column
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': (df.isnull().sum() / len(df) * 100).round(2)
})

# Sort by worst first
missing = missing[missing['missing_count'] > 0].sort_values(
    'missing_percent', ascending=False
)

print(f"Columns with missing data: {len(missing)} out of {len(df.columns)}")
print(f"\nTop 20 worst columns:\n")
print(missing.head(20).to_string())

Columns with missing data: 63 out of 151

Top 20 worst columns:

                                            missing_count  missing_percent
member_id                                          938821           100.00
desc                                               938821           100.00
orig_projected_additional_accrued_interest         936193            99.72
payment_plan_start_date                            935556            99.65
hardship_length                                    935556            99.65
hardship_dpd                                       935556            99.65
hardship_amount                                    935556            99.65
deferral_term                                      935556            99.65
hardship_last_payment_amount                       935556            99.65
hardship_end_date                                  935556            99.65
hardship_start_date                                935556            99.65
hardship_reason                    

In [4]:
# Drop any column that is more than 50% empty — useless for analysis
threshold = 50
cols_to_drop = missing[missing['missing_percent'] > threshold].index.tolist()

print(f"Columns being dropped (>{threshold}% missing): {len(cols_to_drop)}")
print(f"\nDropped columns:\n{cols_to_drop}")

df = df.drop(columns=cols_to_drop)
print(f"\nColumns remaining: {len(df.columns)}")

Columns being dropped (>50% missing): 43

Dropped columns:
['member_id', 'desc', 'orig_projected_additional_accrued_interest', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_amount', 'deferral_term', 'hardship_last_payment_amount', 'hardship_end_date', 'hardship_start_date', 'hardship_reason', 'hardship_status', 'hardship_type', 'hardship_loan_status', 'hardship_payoff_balance_amount', 'settlement_status', 'debt_settlement_flag_date', 'settlement_date', 'settlement_term', 'settlement_percentage', 'settlement_amount', 'sec_app_mths_since_last_major_derog', 'sec_app_revol_util', 'verification_status_joint', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'revol_bal_joint', 'sec_app_collections_12_mths_ex_med', 'sec_app_earliest_cr_line', 'sec_app_fico_range_low', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_fico_range_high', 'sec_app_open_act_il', 'annual_inc_joint', 'dti_joint', 'mths_since_last_record', 'mths_since_rec

In [5]:
# Out of remaining columns keep only what matters for a DA project
keep_cols = [
    'loan_amnt',        # loan amount requested
    'funded_amnt',      # amount actually funded
    'term',             # loan term (36 or 60 months)
    'int_rate',         # interest rate
    'installment',      # monthly payment
    'grade',            # loan grade (A-G)
    'sub_grade',        # loan sub grade
    'emp_length',       # employment length
    'home_ownership',   # rent, own, mortgage
    'annual_inc',       # annual income
    'verification_status', # income verified or not
    'issue_d',          # date loan issued
    'loan_status',      # fully paid, default, current
    'purpose',          # reason for loan
    'addr_state',       # state
    'dti',              # debt to income ratio
    'earliest_cr_line', # earliest credit line date
    'open_acc',         # number of open accounts
    'pub_rec',          # number of derogatory public records
    'revol_bal',        # revolving balance
    'revol_util',       # revolving utilization rate
    'total_pymnt',      # total payment received
    'total_rec_prncp',  # principal received
    'total_rec_int',    # interest received
    'last_pymnt_d',     # last payment date
    'recoveries',       # post charge off gross recovery
]

# Only keep columns that actually exist in the dataframe
keep_cols = [c for c in keep_cols if c in df.columns]

df = df[keep_cols]

print(f"Columns kept: {len(df.columns)}")
print(f"\nColumns: {list(df.columns)}")

Columns kept: 26

Columns: ['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'earliest_cr_line', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 'last_pymnt_d', 'recoveries']


In [6]:
# Check current data types of our 26 columns
print("Current data types:\n")
print(df.dtypes)

Current data types:

loan_amnt              float64
funded_amnt            float64
term                       str
int_rate               float64
installment            float64
grade                      str
sub_grade                  str
emp_length                 str
home_ownership             str
annual_inc             float64
verification_status        str
issue_d                    str
loan_status                str
purpose                    str
addr_state                 str
dti                    float64
earliest_cr_line           str
open_acc               float64
pub_rec                float64
revol_bal              float64
revol_util             float64
total_pymnt            float64
total_rec_prncp        float64
total_rec_int          float64
last_pymnt_d               str
recoveries             float64
dtype: object


In [7]:
# 1. Fix term — remove " months" and convert to integer
df['term'] = df['term'].str.replace(' months', '').str.strip().astype(int)

# 2. Fix int_rate — already float but confirm
df['int_rate'] = pd.to_numeric(df['int_rate'], errors='coerce')

# 3. Fix emp_length — extract numbers from text like "10+ years", "< 1 year"
df['emp_length'] = df['emp_length'].str.replace('+ years', '')\
                                   .str.replace(' years', '')\
                                   .str.replace(' year', '')\
                                   .str.replace('< 1', '0')\
                                   .str.strip()
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')

# 4. Fix issue_d — convert to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# 5. Fix earliest_cr_line — convert to datetime
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# 6. Fix revol_util — remove % and convert to float
df['revol_util'] = df['revol_util'].str.replace('%', '').str.strip()
df['revol_util'] = pd.to_numeric(df['revol_util'], errors='coerce')

print("Data types fixed ✅")
print("\nUpdated data types:\n")
print(df.dtypes)

ValueError: time data "2018-03-01" doesn't match format "%b-%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [8]:
# 1. Fix term — remove " months" and convert to integer
df['term'] = df['term'].str.replace(' months', '').str.strip().astype(int)

# 2. Fix int_rate — already float but confirm
df['int_rate'] = pd.to_numeric(df['int_rate'], errors='coerce')

# 3. Fix emp_length — extract numbers from text like "10+ years", "< 1 year"
df['emp_length'] = df['emp_length'].str.replace('+ years', '')\
                                   .str.replace(' years', '')\
                                   .str.replace(' year', '')\
                                   .str.replace('< 1', '0')\
                                   .str.strip()
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')

# 4. Fix issue_d — mixed formats so use format='mixed'
df['issue_d'] = pd.to_datetime(df['issue_d'], format='mixed')

# 5. Fix earliest_cr_line — mixed formats too
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='mixed')

# 6. Fix revol_util — remove % and convert to float
df['revol_util'] = df['revol_util'].astype(str).str.replace('%', '').str.strip()
df['revol_util'] = pd.to_numeric(df['revol_util'], errors='coerce')

print("Data types fixed ✅")
print("\nUpdated data types:\n")
print(df.dtypes)

AttributeError: Can only use .str accessor with string values, not integer

In [10]:
# 1. Fix term — check if it needs cleaning first
print(f"term sample values: {df['term'].head().tolist()}")
print(f"term dtype: {df['term'].dtype}")

# Only strip text if it's stored as string
if df['term'].dtype == 'object':
    df['term'] = df['term'].str.replace(' months', '').str.strip().astype(int)
else:
    df['term'] = df['term'].astype(int)

# 2. Fix int_rate
df['int_rate'] = pd.to_numeric(df['int_rate'], errors='coerce')

# 3. Fix emp_length
print(f"\nemp_length sample values: {df['emp_length'].head().tolist()}")
print(f"emp_length dtype: {df['emp_length'].dtype}")

if df['emp_length'].dtype == 'object':
    df['emp_length'] = df['emp_length'].str.replace('+ years', '')\
                                       .str.replace(' years', '')\
                                       .str.replace(' year', '')\
                                       .str.replace('< 1', '0')\
                                       .str.strip()
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')

# 4. Fix issue_d — mixed formats
df['issue_d'] = pd.to_datetime(df['issue_d'], format='mixed')

# 5. Fix earliest_cr_line — mixed formats
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line

SyntaxError: unterminated string literal (detected at line 30) (1409098254.py, line 30)

In [12]:
# 1. Fix term
print(f"term dtype: {df['term'].dtype}")
if df['term'].dtype == 'object':
    df['term'] = df['term'].str.replace(' months', '').str.strip().astype(int)
else:
    df['term'] = df['term'].astype(int)

# 2. Fix int_rate
df['int_rate'] = pd.to_numeric(df['int_rate'], errors='coerce')

# 3. Fix emp_length
print(f"emp_length dtype: {df['emp_length'].dtype}")
if df['emp_length'].dtype == 'object':
    df['emp_length'] = df['emp_length'].str.replace('+ years', '')
    df['emp_length'] = df['emp_length'].str.replace(' years', '')
    df['emp_length'] = df['emp_length'].str.replace(' year', '')
    df['emp_length'] = df['emp_length'].str.replace('< 1', '0')
    df['emp_length'] = df['emp_length'].str.strip()
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')

# 4. Fix issue_d
df['issue_d'] = pd.to_datetime(df['issue_d'], format='mixed')

# 5. Fix earliest_cr_line
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='mixed')

# 6. Fix revol_util
print(f"revol_util dtype: {df['revol_util'].dtype}")
if df['revol_util'].dtype == 'object':
    df['revol_util'] = df['revol_util'].str.replace('%', '').str.strip()
df['revol_util'] = pd.to_numeric(df['revol_util'], errors='coerce')

print("Data types fixed")
print(df.dtypes)

term dtype: int64
emp_length dtype: float64
revol_util dtype: float64
Data types fixed
loan_amnt                     float64
funded_amnt                   float64
term                            int64
int_rate                      float64
installment                   float64
grade                             str
sub_grade                         str
emp_length                    float64
home_ownership                    str
annual_inc                    float64
verification_status               str
issue_d                datetime64[us]
loan_status                       str
purpose                           str
addr_state                        str
dti                           float64
earliest_cr_line       datetime64[us]
open_acc                      float64
pub_rec                       float64
revol_bal                     float64
revol_util                    float64
total_pymnt                   float64
total_rec_prncp               float64
total_rec_int                 float64
l

In [13]:
# Standardize loan_status — simplify to 3 clean categories
print("Unique loan_status values before:")
print(df['loan_status'].unique())

# Map all variations to 3 categories
status_map = {
    'Fully Paid': 'Fully Paid',
    'Current': 'Current',
    'Charged Off': 'Defaulted',
    'Late (31-120 days)': 'At Risk',
    'In Grace Period': 'At Risk',
    'Late (16-30 days)': 'At Risk',
    'Does not meet the credit policy. Status:Fully Paid': 'Fully Paid',
    'Does not meet the credit policy. Status:Charged Off': 'Defaulted',
    'Default': 'Defaulted'
}

df['loan_status'] = df['loan_status'].map(status_map)

print("\nUnique loan_status values after:")
print(df['loan_status'].value_counts())

Unique loan_status values before:
<StringArray>
[           'Current',         'Fully Paid', 'Late (31-120 days)',
  'Late (16-30 days)',        'Charged Off',    'In Grace Period',
            'Default']
Length: 7, dtype: str

Unique loan_status values after:
loan_status
Current       689032
Fully Paid    177596
Defaulted      48043
At Risk        24150
Name: count, dtype: int64


In [14]:
print("Missing values before:\n")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill numeric columns with median
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Fill text columns with mode (most common value)
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)

print("\nMissing values after:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nNo missing values remaining ✅" if df.isnull().sum().sum() == 0 else "Some missing values remain")

Missing values before:

emp_length      73858
dti              1646
revol_util       1036
last_pymnt_d     1240
dtype: int64


C:\Users\vakul\AppData\Local\Temp\ipykernel_28104\3417058914.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=['object']).columns



Missing values after:
Series([], dtype: int64)

No missing values remaining ✅


In [15]:
# Remove outliers from key numeric columns using IQR method
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df)
    df = df[(df[column] >= lower) & (df[column] <= upper)]
    after = len(df)
    print(f"{column}: removed {before - after:,} outliers")
    return df

# Apply to key columns
for col in ['annual_inc', 'loan_amnt', 'dti', 'revol_bal']:
    df = remove_outliers(df, col)

print(f"\nFinal dataset rows: {len(df):,}")

annual_inc: removed 45,859 outliers
loan_amnt: removed 24,091 outliers
dti: removed 16,265 outliers
revol_bal: removed 47,400 outliers

Final dataset rows: 805,206


In [16]:
# Remove outliers from key numeric columns using IQR method
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df)
    df = df[(df[column] >= lower) & (df[column] <= upper)]
    after = len(df)
    print(f"{column}: removed {before - after:,} outliers")
    return df

# Apply to key columns
for col in ['annual_inc', 'loan_amnt', 'dti', 'revol_bal']:
    df = remove_outliers(df, col)

print(f"\nFinal dataset rows: {len(df):,}")

annual_inc: removed 21,488 outliers
loan_amnt: removed 470 outliers
dti: removed 1,341 outliers
revol_bal: removed 16,857 outliers

Final dataset rows: 765,050


In [17]:
# Save to processed folder — this is your final analysis-ready dataset
save_path = '../data/processed/lending_club_clean.csv'
df.to_csv(save_path, index=False)

print(f"Clean file saved ✅")
print(f"Location: {save_path}")
print(f"Final shape: {df.shape}")
print(f"\nCleaning Summary:")
print(f"  Original columns: 151")
print(f"  Final columns:    {df.shape[1]}")
print(f"  Final rows:       {df.shape[0]:,}")

Clean file saved ✅
Location: ../data/processed/lending_club_clean.csv
Final shape: (765050, 26)

Cleaning Summary:
  Original columns: 151
  Final columns:    26
  Final rows:       765,050
